# 1. Object-Oriented Programming (OOP)

## Question: What is OOP in Python?

### Topics to Cover

1. What is Object-Oriented Programming (OOP)?
2. What are classes and objects?
3. What are attributes and methods?
4. What are the four main principles of OOP?
   - Encapsulation
   - Inheritance
   - Polymorphism
   - Abstraction
5. How is OOP used in FastAPI?

### Practical Examples
- Creating classes and objects in Python
- Using attributes and methods
- Implementing the four principles of OOP
- Using OOP concepts in a FastAPI application

In [ ]:
class RegisterRequest(pydantic.BaseModel):
    username: str
    password: str

#url has /register/
@app.post("/register/")
def register(
    data: RegisterRequest,
    db: Session = Depends(get_db)
):

    existing_user = (
        db.query(User)
        .filter(User.username == data.username)
        .first()
    )

    if existing_user:
        raise fastapi.HTTPException(
            status_code=400,
            detail="username already exists"
        )

    hashed_password = password_hash.hash(data.password)

    new_user = User(
        username=data.username,
        password=hashed_password
    )

    db.add(new_user)
    db.commit()
    db.refresh(new_user)

    return {
        "message": "user created successfully",
        "username": new_user.username
    }


# Password Hashing in FastAPI

## Question: What is Password Hashing?

Password hashing is the process of converting a password into a hash using a one-way cryptographic algorithm.

Instead of storing plain-text passwords in a database, we store their password hashes.

## Why Is Password Hashing Important?

- Protects users' passwords if the database is compromised.
- Prevents storing passwords in plain text.
- Improves the security of authentication systems.

In [ ]:
from pwdlib import PasswordHash
    hashed_password = password_hash.hash(data.password)

    new_user = User(
        username=data.username,
        password=hashed_password
    )

# Database Configuration in FastAPI

## Question: What is database.py?

The `database.py` file is used to configure the database connection and manage database sessions in a FastAPI application.

When using PostgreSQL with SQLAlchemy, this file typically contains the database URL, engine, session factory, and `get_db()` dependency.

In [ ]:
DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False,
)
class Base(DeclarativeBase):
    pass


def get_db():
    db = SessionLocal()

    try:
        yield db
    finally:
        db.close()

# Models in FastAPI

## Question: What is a Model?

A model defines the structure of data stored in a database.

When using SQLAlchemy ORM, models are Python classes that represent database tables. Their attributes represent columns in those tables.

In a FastAPI application, models are commonly used to define database tables and manage data through SQLAlchemy.

## What Are the Main Responsibilities of Models?

- Define database tables and columns.
- Specify data types and constraints.
- Define relationships between tables.
- Represent database records as Python objects.
- Support CRUD operations (Create, Read, Update, Delete).

In [ ]:
from sqlalchemy import String
from sqlalchemy.orm import Mapped, mapped_column

from database import Base


class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(
        primary_key=True,
        index=True
    )

    username: Mapped[str] = mapped_column(
        String(50),
        unique=True,
        nullable=False,
        index=True
    )

    password: Mapped[str] = mapped_column(
        String(255),
        nullable=False
    )